# Week 6 Exercise — West Africa Price Predictor (ECOWAS)

**Author:** asket  
**Location:** `community-contributions/asket/week6/`

This notebook implements a **price predictor for West Africa** aligned with the *"The Price is Right"* capstone. It is oriented for **ECOWAS** use, with the default currency **XOF** (West African CFA franc).

- **Mode 1 — Single price (XOF):** Predict price in XOF for West Africa / UEMOA reporting.
- **Mode 2 — ECOWAS cross-country:** Same product in **Nigeria (NGN)**, **Ghana (GHS)**, **Senegal (XOF)**, **Côte d'Ivoire (XOF)**.

Designed to be usable by **ECOWAS agencies** (Commission, UEMOA, customs, trade directorates) for price monitoring and reporting.

---

## Week 6 roadmap (main repo)

| Day | Topic | Notebook |
|-----|--------|----------|
| Day 1 | Data Curation | [day1.ipynb](../../day1.ipynb) |
| Day 2 | Data Pre-processing | [day2.ipynb](../../day2.ipynb) |
| Day 3 | Evaluation, Baselines, Traditional ML | [day3.ipynb](../../day3.ipynb) |
| Day 4 | Deep Learning and LLMs | [day4.ipynb](../../day4.ipynb) |
| Day 5 | Fine-tuning a Frontier Model | [day5.ipynb](../../day5.ipynb) |

**Going further:** See the main repo `week6/` for the full capstone (data curation, pre-processing, baselines, neural nets, fine-tuning).

## Setup — Imports and environment

**Dependencies:** `numpy`, `gradio`, `python-dotenv`, `openai`, `scikit-learn`, `ipykernel`. No HuggingFace Hub or week6 `pricer` package; notebook is self-contained.

Run the **first cell below** to install all required packages (once per environment). Then run the imports cell.

In [3]:
# Install all dependencies (run this cell once; then run the rest of the notebook)
%pip install -q numpy gradio python-dotenv openai scikit-learn ipykernel

error: externally-managed-environment

× This environment is externally managed
╰─> To install Python packages system-wide, try brew install
    xyz, where xyz is the package you are trying to
    install.
    
    If you wish to install a Python library that isn't in Homebrew,
    use a virtual environment:
    
    python3 -m venv path/to/venv
    source path/to/venv/bin/activate
    python3 -m pip install xyz
    
    If you wish to install a Python application that isn't in Homebrew,
    it may be easiest to use 'pipx install xyz', which will manage a
    virtual environment for you. You can install pipx with
    
    brew install pipx
    
    You may restore the old behavior of pip by passing
    the '--break-system-packages' flag to pip, or by adding
    'break-system-packages = true' to your pip.conf file. The latter
    will permanently disable this error.
    
    If you disable this error, we STRONGLY recommend that you additionally
    pass the '--user' flag to pip, or set 

In [4]:
import os
import re
from pathlib import Path

import numpy as np
import gradio as gr
from dotenv import load_dotenv
from openai import OpenAI
from sklearn.metrics import mean_absolute_error, r2_score

load_dotenv(override=True)

/Users/franckasket/Library/Python/3.13/lib/python/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


False

In [2]:
# API: OpenRouter (OPENROUTER_API_KEY) or OpenAI (OPENAI_API_KEY). Model: gpt-4o-mini
MODEL = "gpt-4o-mini"
DEFAULT_N = 10  # number of items for evaluation (MAE / R²)
CURRENCY_XOF = "XOF"

if os.environ.get("OPENROUTER_API_KEY"):
    client = OpenAI(
        base_url="https://openrouter.ai/api/v1",
        api_key=os.environ["OPENROUTER_API_KEY"],
    )
    print("Using OpenRouter API")
else:
    client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY", ""))
    print("Using OpenAI API")

Using OpenAI API


## Data — Load prices in XOF

Resolves `week6/human_out.csv` (prices in XOF) when run from repo root or `week6/`. Fallback: West Africa–oriented inline sample in **XOF**.

In [3]:
def _parse_csv_row(row: str):
    """Parse a row '...text...,price' -> (text, price)."""
    idx = row.rfind(",")
    if idx == -1:
        return None
    text = row[:idx].strip().strip('"')
    try:
        price = float(row[idx + 1 :].strip())
    except ValueError:
        return None
    return (text, price)


def load_data_xof():
    """Load (text, price) list in XOF. Tries week6/human_out.csv then fallback sample."""
    # Run from repo root: week6/human_out.csv; from week6/: human_out.csv; from asket/week6/: ../../../human_out.csv
    candidates = [
        Path("week6/human_out.csv"),
        Path("human_out.csv"),
        Path("../human_out.csv"),
        Path("../../human_out.csv"),
        Path("../../../human_out.csv"),
    ]
    for path in candidates:
        if path and path.exists():
            items = []
            with open(path, "r", encoding="utf-8") as f:
                for line in f:
                    line = line.strip()
                    if not line:
                        continue
                    parsed = _parse_csv_row(line)
                    if parsed:
                        items.append({"text": parsed[0], "price": parsed[1]})
            if items:
                print(f"Loaded {len(items)} items from {path} (prices in XOF)")
                return items

    # Fallback: West Africa–oriented sample in XOF
    FALLBACK = [
        {"text": "Title: Rice 25 kg bag (local, parboiled)\nCategory: Food\nDescription: Standard 25 kg bag of parboiled rice, common in West African markets.", "price": 12500.0},
        {"text": "Title: Solar lamp LED 5W\nCategory: Electronics\nDescription: Portable solar rechargeable LED lamp for off-grid lighting.", "price": 3500.0},
        {"text": "Title: Cooking oil 1L (vegetable)\nCategory: Food\nDescription: One litre bottle of vegetable cooking oil.", "price": 1200.0},
        {"text": "Title: Cement 50 kg bag\nCategory: Building\nDescription: Standard 50 kg bag of Portland cement for construction.", "price": 4500.0},
        {"text": "Title: Second-hand smartphone Android\nCategory: Electronics\nDescription: Used Android smartphone, 4G, 2GB RAM.", "price": 35000.0},
        {"text": "Title: School backpack\nCategory: Accessories\nDescription: Durable school backpack for students.", "price": 5500.0},
        {"text": "Title: Mosquito net (treated)\nCategory: Home\nDescription: Insecticide-treated bed net, single/double.", "price": 2500.0},
        {"text": "Title: Motorcycle helmet\nCategory: Automotive\nDescription: Standard safety helmet for motorbike.", "price": 8500.0},
        {"text": "Title: Maize flour 1 kg\nCategory: Food\nDescription: 1 kg pack of maize flour for porridge.", "price": 600.0},
        {"text": "Title: Plastic chairs set of 4\nCategory: Furniture\nDescription: Set of 4 plastic chairs for home or outdoor.", "price": 18000.0},
    ]
    print(f"Using fallback sample: {len(FALLBACK)} items (prices in XOF)")
    return FALLBACK


data_xof = load_data_xof()
print(f"Total items: {len(data_xof)}")

Loaded 100 items from ../../human_out.csv (prices in XOF)
Total items: 100


## Predictor — Single LLM call (XOF)

One LLM call per item; temperature 0; reply parsed for a single numeric price in XOF.

In [4]:
def parse_price_from_reply(reply: str) -> float:
    """Extract a single number from LLM reply (handles '12000', '12 000 XOF', etc.)."""
    if reply is None:
        return 0.0
    reply = str(reply).replace(",", "").replace(" ", "")
    match = re.search(r"[-+]?\d*\.?\d+", reply)
    return float(match.group()) if match else 0.0


def predict_price_xof(text: str) -> float:
    """Predict price in XOF for a product description. Single LLM call, temperature 0."""
    prompt = f"""Estimate the market price in West African CFA francs (XOF) for this product. Reply with a single number only (no currency symbol, no explanation).

Product:
{text}

Price in XOF:"""
    try:
        resp = client.chat.completions.create(
            model=MODEL,
            messages=[{"role": "user", "content": prompt}],
            temperature=0,
        )
        content = resp.choices[0].message.content
        return parse_price_from_reply(content)
    except Exception as e:
        print(f"API error: {e}")
        return 0.0

## Baselines (Day 3 style) — Constant predictor

Constant (mean) predictor; evaluation compares LLM vs baseline using MAE and R² in XOF.

In [5]:
# Constant baseline: predict mean training price (XOF)
mean_price_xof = float(np.mean([d["price"] for d in data_xof]))

def baseline_constant_xof(_text):
    return mean_price_xof

print(f"Constant baseline (mean price XOF): {mean_price_xof:,.0f} XOF")

Constant baseline (mean price XOF): 84 XOF


## Evaluation — MAE and R² in XOF

Evaluate on first `N` items (default 10). Compare LLM predictor vs constant baseline.

In [6]:
def evaluate_predictor(data, predictor, n=DEFAULT_N):
    """Compute MAE and R² in XOF on first n items."""
    subset = data[:n]
    truths = [d["price"] for d in subset]
    guesses = [predictor(d["text"]) for d in subset]
    mae = mean_absolute_error(truths, guesses)
    r2 = r2_score(truths, guesses)
    return mae, r2, truths, guesses


n_eval = DEFAULT_N
mae_baseline, r2_baseline, _, _ = evaluate_predictor(data_xof, baseline_constant_xof, n_eval)
print(f"Constant baseline (first {n_eval} items) — MAE: {mae_baseline:,.0f} XOF  R²: {r2_baseline:.3f}")

Constant baseline (first 10 items) — MAE: 67 XOF  R²: -0.014


In [7]:
# LLM predictor evaluation (first n items — may take a minute)
mae_llm, r2_llm, truths, guesses = evaluate_predictor(data_xof, predict_price_xof, n_eval)
print(f"LLM predictor (first {n_eval} items) — MAE: {mae_llm:,.0f} XOF  R²: {r2_llm:.3f}")
print("\nPer-item (truth vs guess XOF):")
for i in range(min(5, n_eval)):
    print(f"  {truths[i]:,.0f} vs {guesses[i]:,.0f}")

LLM predictor (first 10 items) — MAE: 55,406 XOF  R²: -676267.506

Per-item (truth vs guess XOF):
  120 vs 150,000
  300 vs 45,000
  43 vs 25,000
  55 vs 25,000
  12 vs 25,000


## ECOWAS cross-country comparison

One product → estimated price in **Nigeria (NGN)**, **Ghana (GHS)**, **Senegal (XOF)**, **Côte d'Ivoire (XOF)**.

In [8]:
ECOWAS_COUNTRIES = [
    ("Nigeria", "NGN"),
    ("Ghana", "GHS"),
    ("Senegal", "XOF"),
    ("Côte d'Ivoire", "XOF"),
]


def predict_ecowas_cross_country(text: str) -> dict:
    """Return dict country -> (currency, price) for ECOWAS cross-country comparison."""
    result = {}
    for country, currency in ECOWAS_COUNTRIES:
        prompt = f"""Estimate the typical market price for this product in {country}. Reply with a single number only (no currency symbol, no explanation). Use the local currency: {currency}.

Product:
{text}

Price in {currency}:"""
        try:
            resp = client.chat.completions.create(
                model=MODEL,
                messages=[{"role": "user", "content": prompt}],
                temperature=0,
            )
            content = resp.choices[0].message.content
            price = parse_price_from_reply(content)
            result[country] = (currency, price)
        except Exception as e:
            result[country] = (currency, 0.0)
            print(f"API error for {country}: {e}")
    return result

## Gradio — Two tabs

1. **Single price (XOF):** Input product description → predicted price in XOF.  
2. **Cross-country (ECOWAS):** Input product description → table of prices in Nigeria (NGN), Ghana (GHS), Senegal (XOF), Côte d'Ivoire (XOF).

In [9]:
def gradio_single_xof(description: str) -> str:
    if not (description or description.strip()):
        return "Enter a product description."
    price = predict_price_xof(description.strip())
    return f"{price:,.0f} XOF"


def gradio_cross_country(description: str) -> str:
    if not (description or description.strip()):
        return "Enter a product description."
    result = predict_ecowas_cross_country(description.strip())
    lines = ["| Country | Currency | Price |", "|---------|----------|-------|"]
    for country, (currency, price) in result.items():
        lines.append(f"| {country} | {currency} | {price:,.0f} |")
    return "\n".join(lines)


with gr.Blocks(title="West Africa Price Predictor (ECOWAS)") as demo:
    gr.Markdown("# West Africa Price Predictor (ECOWAS)")
    with gr.Tabs():
        with gr.Tab("Single price (XOF)"):
            inp_xof = gr.Textbox(
                label="Product description",
                placeholder="e.g. Title: Rice 25 kg bag... Category: Food...",
                lines=5,
            )
            out_xof = gr.Textbox(label="Predicted price (XOF)")
            gr.Button("Predict").click(fn=gradio_single_xof, inputs=inp_xof, outputs=out_xof)
        with gr.Tab("Cross-country (ECOWAS)"):
            inp_eco = gr.Textbox(
                label="Product description",
                placeholder="e.g. Title: Solar lamp... Category: Electronics...",
                lines=5,
            )
            out_eco = gr.Markdown(label="Prices by country")
            gr.Button("Compare").click(fn=gradio_cross_country, inputs=inp_eco, outputs=out_eco)

demo.launch()

* Running on local URL:  http://127.0.0.1:7868
* To create a public link, set `share=True` in `launch()`.


---

## How to run

1. Set `OPENROUTER_API_KEY` or `OPENAI_API_KEY` in `.env`.
2. Open this notebook and run all cells.
3. For more data, run from **week6/** so `human_out.csv` is found (or copy it next to the notebook).
4. Run the evaluation cells above to see MAE/R² on the sample; the last cell launches Gradio.

In [ ]:
# Optional: run both evaluations in one go
# evaluate_predictor(data_xof, baseline_constant_xof, n_eval)  # baseline
# evaluate_predictor(data_xof, predict_price_xof, n_eval)      # LLM (slow)